# U-Net（PASCAL VOC 2007を用いたセマンティックセグメンテーション）

---
## 目的
医用画像セグメンテーション向けに提案されたU-Net（Ronneberger et al., 2015）を，PASCAL VOC 2007データセットを用いて実装する．U-Netは，事前学習済みモデルに頼らずスクラッチで学習できるよう設計された，対称なエンコーダ・デコーダ構造を持つ畳み込みネットワークである．エンコーダの各解像度の特徴マップをデコーダにスキップ接続でconcatし，高解像度な情報を保ちながらセグメンテーションマスクを復元する仕組みを理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import VOCSegmentation
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して画素単位のクラスラベル（セグメンテーションマスク）が付与されたデータセットで，学習用（trainval）422枚，評価用（test）210枚の画像から構成されます（矩形領域のみのVOCDetectionと異なり，セグメンテーション用のアノテーションが存在する画像のみに限られるため，画像分類・物体検出のノートブックと比べて画像枚数が少なくなっています）．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCSegmentation`で読み込みます．マスク画像は，各画素の値がクラスID（0が背景，1〜20が物体クラス，255が物体の境界など評価から除外する「無視領域」）になっている画像として与えられます．

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
n_class = len(VOC_CLASSES)  # 21（背景を含む）
IGNORE_INDEX = 255  # 物体境界などの無視領域
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class VOCSegmentationDataset(torch.utils.data.Dataset):
    """VOCSegmentationをラップし，画像とマスクを同じサイズにリサイズしたTensorとして返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCSegmentation(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, mask = self.voc[idx]
        image = self.img_transform(image)
        # マスクは値がそのままクラスIDのため，補間で値が混ざらないよう最近傍補間でリサイズする
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask


batch_size = 8
train_data = VOCSegmentationDataset('trainval')
test_data = VOCSegmentationDataset('test')
print('train:', len(train_data), 'test:', len(test_data))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

image, mask = train_data[0]
print('image:', image.shape, 'mask:', mask.shape, 'mask unique:', torch.unique(mask))

## VOCカラーパレットと可視化用関数
クラスIDの2次元配列（マスク）をそのまま画像として表示しても分かりにくいため，各クラスIDに固有の色を割り当ててカラー画像に変換する関数を用意します．ここでは，VOCdevkitで標準的に使われているビット演算によるカラーパレットの生成方法を使用します．

In [ ]:
def voc_colormap(n=n_class):
    """VOCdevkitで標準的に使われる、クラスIDから固有のRGB色を生成する関数"""
    cmap = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = [r, g, b]
    return cmap


VOC_COLORMAP = voc_colormap()


def decode_segmap(mask):
    """クラスIDのマスク(H, W)を，可視化用のカラー画像(H, W, 3)に変換する"""
    mask = mask.clone()
    mask[mask == IGNORE_INDEX] = 0  # 可視化のため，無視領域は背景色として扱う
    return VOC_COLORMAP[mask.cpu().numpy()]

## U-Net
U-Netは，画像分類ネットワークをバックボーンとして流用するFCNとは異なり，**事前学習済みモデルに一切依存せず，全てのパラメータをスクラッチで学習する**ことを前提に設計されたネットワークです．少数の学習データしか得られない医用画像分野を主なターゲットとしており，本ノートブックで扱うVOC2007のtrainval（422枚）のような小規模データでの学習にも適した構造です．

ネットワーク全体は，入力画像を段階的にダウンサンプリングしながら特徴を抽出する**エンコーダ（収縮パス）**と，特徴マップを段階的にアップサンプリングしながら元の解像度までマスクを復元する**デコーダ（拡大パス）**からなり，形状がアルファベットの「U」に似ていることからU-Netと呼ばれます．

### 畳み込みブロック（DoubleConv）
U-Netのエンコーダ・デコーダは，どちらも「3x3畳み込み→バッチ正規化→ReLU」を2回繰り返す共通のブロックを基本単位として構成されます．原論文では畳み込みにパディングを行わない（unpadded convolution）ため，畳み込みのたびに特徴マップが縮小し，デコーダ側でスキップ接続を行う際に特徴マップをクロップ（中心切り出し）する必要がありました．本ノートブックでは実装を単純化するため，`padding=1`のパディングありの3x3畳み込みを使用し，畳み込みによる特徴マップサイズの縮小自体を起こさない構成にしています（この違いはノートブック末尾の表にまとめます）．

In [ ]:
class DoubleConv(nn.Module):
    """3x3畳み込み(padding=1) -> BN -> ReLU を2回繰り返すブロック"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

### エンコーダ（収縮パス）
エンコーダは，`DoubleConv`と`nn.MaxPool2d`（stride 2）を4段繰り返し，チャンネル数を64→128→256→512と倍増させながら解像度を1/2ずつ縮小していきます．最も深い段（ボトルネック）では，さらに`DoubleConv`でチャンネル数を1024にします．各段のプーリング前の特徴マップは，デコーダのスキップ接続で使用するために保持しておきます．

In [ ]:
class Encoder(nn.Module):
    """4段のダウンサンプリングでチャンネル数を64->128->256->512->1024と倍増させる"""
    def __init__(self, in_channels=3):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.bottleneck = DoubleConv(512, 1024)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        e1 = self.enc1(x)              # (B, 64,  H,    W)
        e2 = self.enc2(self.pool(e1))  # (B, 128, H/2,  W/2)
        e3 = self.enc3(self.pool(e2))  # (B, 256, H/4,  W/4)
        e4 = self.enc4(self.pool(e3))  # (B, 512, H/8,  W/8)
        b = self.bottleneck(self.pool(e4))  # (B, 1024, H/16, W/16)
        return e1, e2, e3, e4, b

### デコーダ（拡大パス）とスキップ接続
デコーダは，`nn.ConvTranspose2d`（kernel_size=2, stride=2）で特徴マップを2倍にアップサンプリングしたのち，**エンコーダの対応する解像度の特徴マップをチャンネル方向にconcat**し，`DoubleConv`でチャンネル数を整えるという処理を4段繰り返します．転置畳み込みのkernel/strideを2に揃えているため，各段のアップサンプリング後の特徴マップはエンコーダ側の対応する特徴マップと解像度が完全に一致し，原論文のようなクロップ処理は不要です．

U-Net++などのネスト構造を持つ拡張と異なり，U-Netのスキップ接続は**各解像度につき1本のみ**（エンコーダの同じ解像度の出力をそのままconcatする）という単純な構成である点に注意してください．

In [ ]:
class Decoder(nn.Module):
    """4段のアップサンプリングと，各段でエンコーダの同解像度特徴をconcatするスキップ接続"""
    def __init__(self, n_class):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)  # 512(up) + 512(skip) -> 512

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)   # 256(up) + 256(skip) -> 256

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)   # 128(up) + 128(skip) -> 128

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)    # 64(up) + 64(skip) -> 64

        self.out_conv = nn.Conv2d(64, n_class, kernel_size=1)  # 1x1畳み込みでクラス数チャンネルに変換

    def forward(self, e1, e2, e3, e4, b):
        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))  # skip connection：同じ解像度のe4をconcat

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))  # skip connection：同じ解像度のe3をconcat

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))  # skip connection：同じ解像度のe2をconcat

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))  # skip connection：同じ解像度のe1をconcat

        return self.out_conv(d1)

### U-Net全体
エンコーダとデコーダを組み合わせ，U-Net全体を構築します．入力サイズがエンコーダ側で4回2分の1にダウンサンプリングされる（$2^4=16$）ため，入力の高さ・幅は16の倍数である必要があります（`IMG_SIZE=256`はこれを満たしています）。

In [ ]:
class UNet(nn.Module):
    def __init__(self, n_class=n_class, in_channels=3):
        super().__init__()
        self.encoder = Encoder(in_channels)
        self.decoder = Decoder(n_class)

    def forward(self, x):
        e1, e2, e3, e4, b = self.encoder(x)
        return self.decoder(e1, e2, e3, e4, b)

## 損失関数
セグメンテーションは，画素ごとのクラス分類問題とみなせるため，`nn.CrossEntropyLoss`をそのまま利用できます．マスクに含まれる無視領域（ラベル255）は`ignore_index`で損失計算から除外します．

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## ネットワークの作成
U-Netを構築し，最適化手法としてAdamを設定します．事前学習済み重みを使わずスクラッチで学習するため，`fcn.ipynb`（事前学習済みバックボーンを使用）よりも大きい学習率`1e-3`を用いています．

In [ ]:
model = UNet(n_class=n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
学習データ（trainval）を用いてU-Netを学習します．VOC2007のセグメンテーション用データは422枚と少ないため，`epoch_num`を多めに設定しています．

In [ ]:
epoch_num = 30

model.train()
for epoch in range(epoch_num):
    sum_loss = 0.0
    count = 0
    for image, mask in train_loader:
        image, mask = image.to(device), mask.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * image.size(0)
        count += image.size(0)

    print(f'epoch: {epoch + 1}, mean loss: {sum_loss / count:.4f}')

## テスト（mIoU評価）
評価用データ（test）に対して，クラスごとのIoU（Intersection over Union）の平均であるmIoU（mean IoU）を求めます．mIoUの算出には，`torchmetrics.classification.MulticlassJaccardIndex`を使用し，無視領域（ラベル255）は`ignore_index`で評価から除外します．

In [ ]:
metric = MulticlassJaccardIndex(num_classes=n_class, ignore_index=IGNORE_INDEX, average='macro').to(device)

model.eval()
with torch.no_grad():
    for image, mask in test_loader:
        image, mask = image.to(device), mask.to(device)
        output = model(image)
        pred = output.argmax(dim=1)
        metric.update(pred, mask)

print(f'mIoU: {metric.compute().item():.4f}')

## セグメンテーション結果の可視化
評価用データから1枚取り出し，入力画像・正解マスク・予測マスクを並べて表示します。

In [ ]:
model.eval()
image, mask = test_data[0]
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).squeeze(0).cpu()

# 表示用に正規化前の画素値へ戻す
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
image_vis = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_vis); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(decode_segmap(mask)); axes[1].set_title('ground truth'); axes[1].axis('off')
axes[2].imshow(decode_segmap(pred)); axes[2].set_title('prediction'); axes[2].axis('off')
plt.show()

## オリジナルのU-Netとの違い

| 項目 | オリジナル論文 (Ronneberger et al., 2015) | 本ノートブック |
| :-- | :-- | :-- |
| 畳み込みのパディング | パディングなし（unpadded convolution）で畳み込みのたびに特徴マップが縮小し，スキップ接続時にエンコーダ側の特徴マップをクロップ（中心切り出し）する | `padding=1`のパディングありの3x3畳み込みを使用し，エンコーダ・デコーダの特徴マップが常に同じ解像度になるためクロップ不要 |
| 正規化層 | バッチ正規化なし | 各畳み込みの後に`nn.BatchNorm2d`を追加し，学習を安定化 |
| 出力サイズ | 入力よりも小さくなる（パディングなしのため） | 入力と完全に同じ解像度（256x256） |
| クラス数の重み付け | 医用画像特有の重み付き損失（境界付近を重視するweight map）を使用 | 通常の`nn.CrossEntropyLoss`（`ignore_index`のみ使用） |
| データ拡張 | 弾性変形（elastic deformation）などを多用 | 固定サイズへのリサイズのみ（データ拡張なし） |
| 学習データ量 | 医用画像（細胞境界セグメンテーション）の数十枚規模 | VOC2007のtrainvalのみ（422枚，20クラス+背景） |

## 課題

1. `epoch_num`や学習率を変更し，mIoUがどのように変化するか確認してください．
2. スキップ接続（デコーダでのconcat）を取り除いた場合，mIoUや予測結果の輪郭がどのように変化するか確認し，スキップ接続の役割を考察してください．
3. `padding=1`の畳み込みを`padding=0`（パディングなし）に変更すると，エンコーダとデコーダの特徴マップのサイズにどのような差が生じるか，実際に確認してください．